# 02 — Degradations gallery

Shows one face under every degradation the project defines (`jpeg`, `blur`,
`downscale`, `noise`, `screen`), at severities 1 (mild) to 5 (severe). This is
the same code used for training augmentation (`augment.degradation` in the
config) and for robustness evaluation
(`python -m deepfake_kyc.evaluate --degradation all`), so the gallery is a
direct picture of what those settings actually do to an image.


In [ ]:
from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "deepfake_kyc").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Open this notebook from inside the "
        "project (e.g. from notebooks/) and select the .venv kernel."
    )


PROJECT_ROOT = find_project_root()

from deepfake_kyc.config import load_config
from deepfake_kyc.data import make_dataset

cfg = load_config(PROJECT_ROOT / "configs" / "base.yaml")
DATA_ROOT = PROJECT_ROOT / cfg["data"]["root"]
TRAIN_DIR = DATA_ROOT / "train"

if TRAIN_DIR.is_dir():
    import random

    from PIL import Image

    ds = make_dataset(TRAIN_DIR)
    sample_path, _ = random.Random(0).choice(ds.samples)
    base_image = Image.open(sample_path).convert("RGB")
    print(f"Using a real sample image: {sample_path}")
else:
    import numpy as np
    from PIL import Image

    rng = np.random.default_rng(0)
    base_image = Image.fromarray(rng.integers(0, 255, (256, 256, 3), dtype="uint8"))
    print(
        f"Dataset not found under {DATA_ROOT}; using a synthetic placeholder image instead. "
        "See docs/03_data.md to download the real dataset."
    )


## The five degradations

Each one models a real way a face image degrades before it reaches a KYC
pipeline:

- **`jpeg`** — lossy compression, like an image sent through a messaging app
- **`blur`** — an out-of-focus or low-quality camera
- **`downscale`** — a low-resolution upload, shrunk then stretched back
- **`noise`** — sensor noise from poor lighting
- **`screen`** — a photo taken of a screen (moiré stripes, colour cast, slight
  blur and compression)


In [ ]:
import matplotlib.pyplot as plt

from deepfake_kyc.degradations import DEGRADATIONS

severities = [1, 2, 3, 4, 5]
names = list(DEGRADATIONS)

fig, axes = plt.subplots(
    len(names), len(severities) + 1,
    figsize=(2.1 * (len(severities) + 1), 2.1 * len(names)),
)
for row, name in enumerate(names):
    axes[row, 0].imshow(base_image)
    axes[row, 0].set_ylabel(name, fontsize=11, rotation=0, labelpad=45, va="center")
    axes[row, 0].set_xticks([])
    axes[row, 0].set_yticks([])
    if row == 0:
        axes[row, 0].set_title("original")
    for col, severity in enumerate(severities, start=1):
        degraded = DEGRADATIONS[name](severity)(base_image)
        axes[row, col].imshow(degraded)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        if row == 0:
            axes[row, col].set_title(f"severity {severity}")
fig.suptitle("One face under every degradation, severities 1 (mild) to 5 (severe)")
fig.tight_layout()


## Where these are used

- Set `augment.degradation: true` in an experiment config to apply a random
  one of these during training (see `configs/experiments/06_effnet_robust.yaml`).
- Run `python -m deepfake_kyc.evaluate --checkpoint <ckpt> --degradation all`
  to measure accuracy at every severity on the held-out test set — those are
  the numbers [03_results_charts.ipynb](03_results_charts.ipynb) charts.

## Next

[03_results_charts.ipynb](03_results_charts.ipynb) loads evaluation CSVs and
plots how accuracy holds up under each of these degradations.
